# Play Store data analysis

Task context: Explore application records, clean duplicates and prices, filter educational apps and summarize price, rating and review counts by category and app type. Demonstrates DataFrame selection, missing-value checks, string operations and pivot tables.

Requires a local `playstore.csv`, which is not included. Saved outputs come from an earlier run. The source of the original notebook and dataset needs manual confirmation.

In [ ]:
import pandas as pd
import numpy as np

## Load application records
Read the local CSV into a DataFrame.

In [2]:
playstore = pd.read_csv("playstore.csv", index_col=0)

In [3]:
playstore.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## Inspect the first and last records
Keep three rows from each end of the table.

In [4]:
playstore = pd.read_csv("playstore.csv", index_col=0)

In [5]:
data_head = playstore.head(3)
data_tail = playstore.tail(3)

## Table dimensions
Read the row and column counts from the DataFrame shape.

In [6]:
n_row, n_col = playstore.shape

## Distinct applications
Count unique values in the application name column.

In [7]:
playstore.App.nunique()

9659

## Missing ratings
Count records with no application rating.

In [8]:
rating_missing = playstore.Rating.isnull().sum()

In [9]:
rating_missing

1474

## Export a selected subset
Combine row slices and select application, size, genre and version columns for CSV export.

In [10]:
# отберем нужные строки датафрейма

temp = pd.concat([playstore[:3], playstore[5:8], playstore[15:19]])

In [11]:
# отберем нужные колонки

result = temp[["App", "Size", "Genres", "Current Ver"]]

In [12]:
result

,App,Size,Genres,Current Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,19M,Art & Design,1.0.0
1,Coloring book moana,14M,Art & Design;Pretend Play,2.0.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",8.7M,Art & Design,1.2.4
5,Paper flowers instructions,5.6M,Art & Design,1.0
6,Smoke Effect Photo Maker - Smoke Editor,19M,Art & Design,1.1
7,Infinite Painter,29M,Art & Design,6.1.61.1
15,Learn To Draw Kawaii Characters,2.7M,Art & Design,NaN
16,Photo Designer - Write your name with shapes,5.5M,Art & Design,3.1
17,350 Diy Room Decor Ideas,17M,Art & Design,1.0
18,FlipaClip - Cartoon animation,39M,Art & Design,2.2.5


In [13]:
# сохраним результат

result.to_csv("5_result.csv")

## Remove duplicate names
Create a table with one record per application and a new positional index.

In [14]:
unique_playstore = playstore.drop_duplicates(subset=['App']).reset_index(drop=True)

## Normalize column names
Use lowercase names and replace spaces with underscores.

In [15]:
playstore.columns = playstore.columns.str.replace(' ', '_').str.lower()

## Free and paid application shares
Remove duplicate application names and calculate relative frequencies by type.

In [16]:
playstore = playstore.drop_duplicates(subset=['app']).reset_index(drop=True) # убираем дубликаты 

In [17]:
playstore.type.value_counts(normalize=True).round(2)

Free    0.92
Paid    0.08
Name: type, dtype: float64

## Filter educational applications
Select EDUCATION records with more than 1,000 reviews.

In [18]:
education_playstore = playstore[(playstore.category == 'EDUCATION')  & (playstore.reviews > 1000)].reset_index(drop=True)

## Convert prices to numbers
Remove the dollar sign and convert price strings to floating-point values.

In [19]:
playstore["price"] = playstore.price.apply(lambda x:x.replace('$','')).astype("float")

## Category and type summary
Calculate mean price, rating and review count, round the results and export the summary.

In [21]:
# первым действием собираем таблицу, затем переименовываем колонки

result = pd.pivot_table(playstore, values=['reviews', 'rating', 'price'], index=['category', 'type'],
                    aggfunc={'rating': np.mean,
                            'reviews': np.mean,
                            'price': np.mean}).\
                    rename(columns={'reviews': 'mean_reviews',
                                    'rating': 'mean_rating',
                                    'price': 'mean_price'})

In [22]:
# округляем колонки "mean_price", "mean_reviews" до двух знаков

result[["mean_price", "mean_reviews"]] = result[["mean_price", "mean_reviews"]].round(2) 

In [23]:
# округляем колонку "mean_rating" до одного знака

result["mean_rating"] = result["mean_rating"].round(1)

In [24]:
result

mean_price  mean_rating  mean_reviews
category          type                                       
ART_AND_DESIGN    Free        0.00          4.3      23230.11
                  Paid        1.99          4.7        722.00
AUTO_AND_VEHICLES Free        0.00          4.2      14140.28
                  Paid        4.49          4.6       1387.67
BEAUTY            Free        0.00          4.3       7476.23
...                            ...          ...           ...
TRAVEL_AND_LOCAL  Paid        4.16          4.1       1506.08
VIDEO_PLAYERS     Free        0.00          4.0     424347.18
                  Paid        2.62          4.1       3341.75
WEATHER           Free        0.00          4.2     171249.62
                  Paid        4.05          4.4      17055.12

[63 rows x 3 columns]

In [25]:
result.to_csv("11_result.csv")